# R3.09 - Cryptographie et Sécurité
## TP 5 : Problème du logarithme discret & Chiffre d'El Gamal
**Enseignant :** M. Bruno Colombel  
**Étudiant :** Mamadou-Bailo BARRY (BUT2 Informatique)

---
# 1. Problème du logarithme discret

### 1.1 Fonction indicatrice d’Euler (Rappels)
- Un entier $a$ est inversible modulo $n$ si et seulement si $\text{pgcd}(a, n) = 1$.
- Si $p$ est premier, alors $\varphi(p) = p - 1$ (tout élément non nul de $\mathbb{Z}/p\mathbb{Z}$ est inversible).
- **Théorème d'Euler / Petit théorème de Fermat :** Si $a$ et $n$ sont premiers entre eux, alors $a^{\varphi(n)} \equiv 1 \pmod n$.  
  En conséquence, l'inverse de $a$ modulo $n$ est donné par :  
  $$a^{-1} \equiv a^{\varphi(n) - 1} \pmod n$$
  En particulier, si $p$ est premier : $a^{-1} \equiv a^{p-2} \pmod p$.

### 1.2 Racine primitive — exponentiation rapide
On utilise l'algorithme des carrés (exponentiation rapide récursive) fourni dans le sujet pour calculer efficacement $a^x \pmod n$ :

In [1]:
def puissance(a, x, n):
    """Méthode récursive, rapide de calcul de a^x mod n (algorithme des carrés)"""
    if x == 0:
        return 1
    elif x == 1:
        return a
    elif x % 2 == 0:
        return puissance(a**2 % n, x // 2, n) % n
    elif x != 1:
        return a * puissance(a**2 % n, (x - 1) // 2, n) % n

def cycle(a, p):
    """Retourne la liste des puissances a**0, a**1, a**2, ... a**(p-1) modulo p"""
    resultat = []
    for d in range(p - 1):
        resultat.append(puissance(a, d, p))
    return resultat

---
## Exercice 1 : Analyse des cycles modulo 31
On étudie le comportement de la fonction $x \mapsto a^x \pmod{31}$ pour $a \in \{2, 3, 4, 5\}$.

In [2]:
p = 31
for a in [2, 3, 4, 5]:
    c = cycle(a, p)
    print(f"Cycle pour a = {a} (longueur {len(c)}, {len(set(c))} valeurs uniques) :\n{c}\n")

Cycle pour a = 2 (longueur 30, 5 valeurs uniques) :
[1, 2, 4, 8, 16, 1, 2, 4, 8, 16, 1, 2, 4, 8, 16, 1, 2, 4, 8, 16, 1, 2, 4, 8, 16, 1, 2, 4, 8, 16]

Cycle pour a = 3 (longueur 30, 30 valeurs uniques) :
[1, 3, 9, 27, 19, 26, 16, 17, 20, 29, 25, 13, 8, 24, 10, 30, 28, 22, 4, 12, 5, 15, 14, 11, 2, 6, 18, 23, 7, 21]

Cycle pour a = 4 (longueur 30, 5 valeurs uniques) :
[1, 4, 16, 2, 8, 1, 4, 16, 2, 8, 1, 4, 16, 2, 8, 1, 4, 16, 2, 8, 1, 4, 16, 2, 8, 1, 4, 16, 2, 8]

Cycle pour a = 5 (longueur 30, 3 valeurs uniques) :
[1, 5, 25, 1, 5, 25, 1, 5, 25, 1, 5, 25, 1, 5, 25, 1, 5, 25, 1, 5, 25, 1, 5, 25, 1, 5, 25, 1, 5, 25]


### Réponses aux questions de l'Exercice 1 :
1. **Affichage :** Les cycles sont affichés ci-dessus pour $a = 2, 3, 4, 5$ modulo $31$.
2. **Pour $a = 2$ :**  
   La suite des puissances boucle sur seulement **5 valeurs** : $[1, 2, 4, 8, 16]$.  
   Par conséquent, la fonction $x \mapsto 2^x \pmod{31}$ n'atteint que 5 éléments sur les 30 éléments de $(\mathbb{Z}/31\mathbb{Z})^*$. Elle n'est donc **pas injective**, et donc **pas bijective**.  
   On observe le même comportement pour :
   - $a = 4$ : cycle de longueur 5 ($[1, 4, 16, 2, 8]$) ;  
   - $a = 5$ : cycle de longueur 3 ($[1, 5, 25]$).
3. **Valeur bijective :**  
   Parmi les valeurs testées, la seule valeur pour laquelle $x \mapsto a^x \pmod{31}$ est bijective est **$a = 3$**.  
   Le cycle généré contient l'intégralité des 30 valeurs distinctes de $(\mathbb{Z}/31\mathbb{Z})^*$. On dit que **3 est une racine primitive modulo 31**.

---
## Exercice 2 : Fonction `primitive(a, p)`
D'après la Définition 3 du cours :  
$a$ est une racine primitive modulo $p$ si et seulement si pour tout $d$ tel que $1 \le d < p - 1$, $a^d \not\equiv 1 \pmod p$.

In [3]:
def primitive(a, p):
    """Teste si a est une racine primitive de p avec p premier."""
    val = 1
    for d in range(1, p - 1):
        val = (val * a) % p
        if val == 1:
            return False
    return True

In [4]:
primitive(3, 31)

True

In [5]:
primitive(4, 31)

False

---
## Exercice 3 : Recherche et liste des racines primitives

In [6]:
def racine_primitive(p):
    """Recherche la plus petite racine primitive de p (p premier)."""
    a = 2
    while not primitive(a, p):
        a += 1
    return a

def liste_racines_primitives(p):
    """Retourne la liste complète des racines primitives de p (p premier)."""
    return [a for a in range(2, p) if primitive(a, p)]

In [7]:
racine_primitive(4999)

3

In [8]:
racine_primitive(1000021)

11

In [9]:
liste_racines_primitives(31)

[3, 11, 12, 13, 17, 21, 22, 24]

> **Remarque pédagogique sur $p = 1000021$ :**  
> Mathématiquement, $1000021 = 11 \times 90911$ n'est pas premier ($11$ est un diviseur). Comme $\text{pgcd}(11, 1000021) = 11 \ne 1$, les puissances $11^d \pmod{1000021}$ sont toujours des multiples de $11$ et n'atteignent jamais la valeur $1$. La boucle de `primitive(11, 1000021)` se termine donc sans jamais croiser $1$, renvoyant `True`. C'est exactement ce qui explique la sortie `11` obtenue dans l'exemple du sujet !

---
# 2. Chiffre d'El Gamal

### Principe du chiffrement asymétrique d'El Gamal (1984) :
1. **Génération des clés (Alice) :**
   - Choisit un grand nombre premier $p$ et une racine primitive $g$ modulo $p$.
   - Choisit une clé privée secrète $a \in \{1, \dots, p - 2\}$.
   - Calcule $A = g^a \pmod p$.
   - Clé publique : $(p, g, A)$ ; Clé privée : $a$.

2. **Chiffrement (Bob) :**
   - Pour envoyer un message $m \in \{0, \dots, p - 1\}$, Bob tire un aléa $b \in \{1, \dots, p - 2\}$.
   - Calcule le secret partagé $K = A^b \pmod p$.
   - Calcule le chiffré $c = m \cdot K$ et $B = g^b \pmod p$.
   - Envoie le cryptogramme $(c, B)$ à Alice.

3. **Déchiffrement (Alice) :**
   - Calcule $B^a \pmod p = (g^b)^a = g^{ab} = (g^a)^b = A^b \pmod p = K$.
   - Retrouve le clair : $m = c \times (B^a)^{-1} \pmod p$.

---
## Exercice 4 : Génération des clés `clefs(k)`

In [10]:
from random import randint
from sympy import randprime

def clefs(k):
    """
    Génère une paire de clés El Gamal avec un module p compris entre 2^k et 2^(k+1) - 1.
    Retourne (public_key, private_key) avec public_key = (p, g, A) et private_key = a.
    """
    p = randprime(2**k, 2**(k + 1))
    g = racine_primitive(p)
    a = randint(1, p - 2)
    A = puissance(g, a, p)
    public_key = (p, g, A)
    private_key = a
    return public_key, private_key

In [11]:
k = 16
public_key, private_key = clefs(k)
print("Clé publique (p, g, A) :", public_key)
print("Clé privée a :", private_key)

Clé publique (p, g, A) : (80363, 2, 66518)
Clé privée a : 64972


---
## Exercice 5 : Chiffrement `chiffrer(m, public_key)`

In [12]:
def chiffrer(m, public_key):
    """
    Chiffre un entier m compris entre 0 et p - 1 avec la clé publique (p, g, A).
    Retourne le couple (c, B).
    """
    p, g, A = public_key
    b = randint(1, p - 2)
    B = puissance(g, b, p)
    c = m * puissance(A, b, p)
    return c, B

In [13]:
(p, g, A) = public_key
m = randint(0, p - 1)
c, B = chiffrer(m, public_key)
print(f"Message clair m = {m}")
print(f"Cryptogramme (c, B) = ({c}, {B})")

Message clair m = 2220
Cryptogramme (c, B) = (161249700, 44765)


---
## Exercice 6 : Déchiffrement `dechiffrer(c, B, private_key)`

In [14]:
def dechiffrer(c, B, private_key, p=None):
    """
    Déchiffre le couple (c, B) avec la clé privée a.
    Formule : m = c * (B^a)^(-1) mod p.
    D'après le Théorème 2 : (B^a)^(-1) = (B^a)^(p-2) mod p.
    """
    if isinstance(p, (tuple, list)):
        p = p[0]
    elif p is None:
        p = globals().get('p', None)
        if p is None:
            raise ValueError("Le module p est requis pour déchiffrer")
            
    a = private_key
    Ba = puissance(B, a, p)
    inv_Ba = puissance(Ba, p - 2, p)
    m = (c * inv_Ba) % p
    return m

In [15]:
res = dechiffrer(c, B, private_key)
print(f"Déchiffré : {res}")
res == m

True

---
## Exercice 7 : Encodage et Décodage de texte
Pour chiffrer du texte, chaque caractère est converti en son code ASCII (`ord(c)`), formaté sur 3 chiffres (`065`, `111`...), puis concaténé en un grand entier.

In [16]:
def encode(texte):
    """Convertit un texte en grand entier via ses codes ASCII sur 3 chiffres."""
    chaine = ""
    for c in texte:
        chaine += str(ord(c)).zfill(3)
    return int(chaine) if chaine else 0

def decode(nombre):
    """Reconstitue le texte à partir du grand entier en utilisant % 1000 et // 1000."""
    texte = ""
    while nombre > 0:
        code = nombre % 1000
        texte = chr(code) + texte
        nombre = nombre // 1000
    return texte

In [17]:
for txt in ["Bob", "Alice", "Cryptographie BUT2"]:
    enc = encode(txt)
    dec = decode(enc)
    print(f"Test : '{txt}' -> {enc} -> '{dec}' (OK: {dec == txt})")
    assert dec == txt

Test 1 : 'Bob' -> 66111098 -> 'Bob' (OK: True)
Test 2 : 'Alice' -> 65108105099101 -> 'Alice' (OK: True)
Test 3 : 'Cryptographie BUT2' -> 67114121112116111103114097112104105101032066085084050 -> 'Cryptographie BUT2' (OK: True)


---
## Exercice 8 : Simulation d'échange de message secret entre voisins

In [18]:
# Alice génère une clé publique avec k=24 pour avoir un module p > m
cle_pub_Alice, cle_priv_Alice = clefs(24)
p_Alice, g_Alice, A_Alice = cle_pub_Alice
print(f"1. Alice publie sa clé publique : {cle_pub_Alice}")

# Bob prépare son message court
message_secret = "OK"
m_entier = encode(message_secret)
print(f"2. Message secret de Bob : '{message_secret}'")
print(f"   Encodé en entier m = {m_entier} (vérification m < p : {m_entier < p_Alice})")

# Bob chiffre avec la clé publique d'Alice
c_envoye, B_envoye = chiffrer(m_entier, cle_pub_Alice)
print(f"3. Cryptogramme envoyé par Bob (c, B) : ({c_envoye}, {B_envoye})")

# Alice reçoit le cryptogramme et le déchiffre avec sa clé privée
m_dechiffre = dechiffrer(c_envoye, B_envoye, cle_priv_Alice, p_Alice)
texte_retrouve = decode(m_dechiffre)
print(f"4. Alice déchiffre le cryptogramme : entier retrouvé = {m_dechiffre}")
print(f"5. Alice décode le texte : '{texte_retrouve}'")
print(f"=> Échange réussi avec succès : {texte_retrouve == message_secret}")
assert texte_retrouve == message_secret

1. Alice publie sa clé publique : (14732159, 2, 8031525)
2. Message secret de Bob : 'OK'
   Encodé en entier m = 79075 (vérification m < p : True)
3. Cryptogramme envoyé par Bob (c, B) : (111100025000, 3105553)
4. Alice déchiffre le cryptogramme : entier retrouvé = 79075
5. Alice décode le texte : 'OK'
=> Échange réussi avec succès : True


---
## Exercice 9 (Bonus) : Gestion des textes longs par découpage en blocs
Pour chiffrer des textes de taille quelconque, on découpe le texte en blocs de taille inférieure au module $p$.

In [19]:
def chiffrer_texte_long(texte, public_key):
    """
    Découpe un texte long en plusieurs blocs pour que chaque bloc soit encodé par un entier < p,
    puis chiffre chaque bloc individuellement.
    """
    p, g, A = public_key
    # Chaque caractère prend 3 chiffres décimaux.
    taille_bloc = max(1, (len(str(p)) - 1) // 3)
    blocs_chiffres = []
    for i in range(0, len(texte), taille_bloc):
        morceau = texte[i:i + taille_bloc]
        m_i = encode(morceau)
        c_i, B_i = chiffrer(m_i, public_key)
        blocs_chiffres.append((c_i, B_i))
    return blocs_chiffres

def dechiffrer_texte_long(blocs_chiffres, private_key, p):
    """
    Déchiffre chaque bloc chiffré et reconstitue le message complet.
    """
    texte_reconstitue = ""
    for c_i, B_i in blocs_chiffres:
        m_i = dechiffrer(c_i, B_i, private_key, p)
        texte_reconstitue += decode(m_i)
    return texte_reconstitue

In [20]:
message_long = "Félicitations ! Le protocole asymétrique d'El Gamal fonctionne parfaitement sur texte long."
chiffres = chiffrer_texte_long(message_long, cle_pub_Alice)
texte_dechiffre = dechiffrer_texte_long(chiffres, cle_priv_Alice, p_Alice)

print(f"Message long original ({len(message_long)} caractères) :\n{repr(message_long)}\n")
print(f"Nombre de blocs chiffrés : {len(chiffres)}")
print(f"Déchiffré : {repr(texte_dechiffre)}")
print(f"Test Bonus réussi : {texte_dechiffre == message_long}")
assert texte_dechiffre == message_long

Message long original (89 caractères) :
'Félicitations ! Le protocole asymétrique d'El Gamal fonctionne parfaitement sur texte long.'

Nombre de blocs chiffrés : 45
Déchiffré : 'Félicitations ! Le protocole asymétrique d'El Gamal fonctionne parfaitement sur texte long.'
Test Bonus réussi : True
